## Сбор и подготовка данных

### Загрузка и подготовка датасета

In [100]:
from src.data_utils import read_texts_from_file
from src.data_utils import clean_texts
from src.data_utils import save_texts_to_file

texts_filename = 'data/tweets.txt'
processed_texts_filename = 'data/tweets_processed.txt'

raw_texts = read_texts_from_file(texts_filename)

print(f"\nВыборка необработанных текстов")
print("-" * 90)
print('\n'.join(map(str, raw_texts[:5])))
print("-" * 90)

texts = clean_texts(raw_texts)
save_texts_to_file(texts, processed_texts_filename)

print(f"\nВыборка очищенных текстов")
print("-" * 90)
print('\n'.join(map(str, texts[:5])))
print("-" * 90)

Загружено 1600498 строк из файла.

Выборка необработанных текстов
------------------------------------------------------------------------------------------
@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D
is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!
@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds
my whole body feels itchy and like its on fire
@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there.
------------------------------------------------------------------------------------------
✅ Успешно сохранено 1600498 текстов в файл: data/tweets_processed.txt

Выборка очищенных текстов
------------------------------------------------------------------------------------------
- awww, thats a bummer. you shoulda got david carr of third day to do it. ;d
is upset th

In [101]:
#урезаем датасет для работы локально
texts = texts[:1000] 

### Токенизация

In [102]:
from collections import Counter
tokenized_texts = [text.split() for text in texts]

all_tokens = [token for text in tokenized_texts for token in text]
word_counter = Counter(all_tokens)

word_to_idx = {word: idx + 1 for idx, word in enumerate(word_counter.keys())}
word_to_idx['<PAD>'] = 0
word_to_idx['<UNK>'] = len(word_to_idx)
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

vocab_size = len(word_to_idx)
print(f"Размер словаря: {vocab_size}")

Размер словаря: 3838


### Подготовка датасетов и даталодеров

In [103]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from src.token_dataset import TokenPredictionDataset
from src.token_dataset import ValDataset
from src.token_dataset import collate_fn


train_texts, val_test_texts = train_test_split(texts, test_size = 0.2, random_state = 42)
val_texts, test_texts = train_test_split(val_test_texts, test_size = 0.5, random_state = 42)

train_dataset = TokenPredictionDataset(train_texts, word_to_idx)
val_dataset = ValDataset(val_texts, word_to_idx)
test_dataset = ValDataset(test_texts, word_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

## Обучение

### Инициализация параметров обучения

In [108]:
import torch
import torch.nn as nn
from rouge_score import rouge_scorer
from src.lstm_model import LSTMNextToken
from src.next_token_utils import tensor_to_text
from src.next_token_utils import generate_next_word
from src.next_token_utils import generate_text
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

embed_dim = 256
embed_dim = 250
hidden_dim = 256
hidden_dim = 128
num_layers = 2
num_layers = 1
lr = 0.01
num_epochs = 20
print_every = 1

model = LSTMNextToken(vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True) #точно ли нужны все 


Используется устройство: cpu


### Процесс обучения

In [112]:
model.train()
best_val_acc = 0.0
print_every = 1


for epoch in range(num_epochs):
    # =======Обучение
    model.train()
    total_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        #accuracy
        _, predicted = outputs.max(1)
        train_total += y_batch.size(0)
        train_correct += (predicted == y_batch).sum().item()

    
    avg_train_loss = total_loss / len(train_loader)
    train_acc = train_correct / train_total

    # ====Валидация 
    if (epoch + 1) % print_every == 0:
        model.eval()
        val_correct = 0
        val_total = 0
        val_rouge1 = []
        val_rougeL = []

        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val = X_val.to(device)
                y_val = y_val.to(device)

                outputs = model(X_val)
                _, predicted = outputs.max(1)
                val_total += y_val.size(0)
                val_correct += (predicted == y_val).sum().item()

                # Преобразуем в текст
                contexts = tensor_to_text(X_val, idx_to_word, truncate_at_pad=True)
                targets = [idx_to_word.get(t.item(), "<UNK>") for t in y_val]
                preds = [idx_to_word.get(p.item(), "<UNK>") for p in predicted]

                # Генерация для ROUGE
                generated = []
                for ctx in contexts:
                    gen_word = generate_next_word(ctx, model, word_to_idx, idx_to_word)
                    generated.append(gen_word)

                # Считаем ROUGE 
                for gen, ref in zip(generated, targets):
                    scores = scorer.score(ref, gen)
                    val_rouge1.append(scores['rouge1'].fmeasure)
                    val_rougeL.append(scores['rougeL'].fmeasure)

        val_acc = val_correct / val_total


        if (epoch + 1) % 4 == 0:
            print(f"\nВалидация — Эпоха {epoch+1}")
            print(f"{'Контекст':<25} | {'Истинное':<15} | {'Предсказано':<15} | {'Сгенерировано':<15}")
            print("-" * 90)
            for i in range(min(10, len(contexts))):
                print(f"{contexts[i]:<25} | {targets[i]:<15} | {preds[i]:<15} | {generated[i]:<15}")
            

        # --- Вывод метрик ---
        print(f"\nЭпоха [{epoch+1}/{num_epochs}]")
        print(f"Loss (train): {avg_train_loss:.4f} | Accuracy (train): {train_acc:.4f}")
        print(f"Accuracy: {val_acc:.4f}")
        print(f"ROUGE-1 F1: {np.mean(val_rouge1):.4f} | ROUGE-L F1: {np.mean(val_rougeL):.4f}")



        # Сохранение лучшей модели (по accuracy) тк rouge плохо меняется
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_lstm_model.pth")
            print(f"✅ Модель сохранена: accuracy = {val_acc:.4f}")

model.train()  
print("\nОбучение завершено.")


Эпоха [1/20]
Loss (train): 7.2335 | Accuracy (train): 0.0321
Accuracy: 0.0205
ROUGE-1 F1: 0.0145 | ROUGE-L F1: 0.0145
✅ Модель сохранена: accuracy = 0.0205

Эпоха [2/20]
Loss (train): 6.8125 | Accuracy (train): 0.0375
Accuracy: 0.0392
ROUGE-1 F1: 0.0324 | ROUGE-L F1: 0.0324
✅ Модель сохранена: accuracy = 0.0392

Эпоха [3/20]
Loss (train): 6.6489 | Accuracy (train): 0.0409
Accuracy: 0.0375
ROUGE-1 F1: 0.0230 | ROUGE-L F1: 0.0230

Валидация — Эпоха 4
Контекст                  | Истинное        | Предсказано     | Сгенерировано  
------------------------------------------------------------------------------------------
please tell me thats somewhere close to | california!!!   | be              | wait           
please tell me thats somewhere close to california!!! | lol!            | 40              | and            
it                        | was             | i               | -              
it was                    | a               | to              | a              
it was a     

## Оценка на тестовом датасете

In [122]:
model.eval()
test_loss = 0.0
correct = 0
total = 0
rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

with torch.no_grad():
    for input_seq, target_token in test_loader:
        input_seq = input_seq.to(device)
        target_token = target_token.to(device)

        # Прямой проход
        outputs = model(input_seq)
        loss = criterion(outputs, target_token)
        test_loss += loss.item()

        # Accuracy
        _, predicted = outputs.max(1)
        total += target_token.size(0)
        correct += (predicted == target_token).sum().item()

        # ROUGE 
        input_texts = tensor_to_text(input_seq, idx_to_word, truncate_at_pad=True)
        target_words = [idx_to_word.get(t.item(), "<UNK>") for t in target_token]

        generated_texts = []
        for context in input_texts:
            gen_word = generate_text(
                context=context,
                model=model,
                word_to_idx=word_to_idx,
                idx_to_word=idx_to_word,
                max_length=1
            )
            
            generated_texts.append(gen_word)
        
        print(f"{'Контекст':<25} | {'Истинное':<15} | {'Предсказано':<15} | {'Сгенерировано':<15}")
        print("-" * 90)
        for i in range(min(10, len(contexts))):
            print(f"{contexts[i]:<25} | {target_words[i]:<15} | {generated_texts[i]:<15}")

        # Считаем ROUGE
        for gen, ref in zip(generated_texts, target_words):
            scores = scorer.score(ref, gen)
            rouge1_scores.append(scores['rouge1'].fmeasure)
            rouge2_scores.append(scores['rouge2'].fmeasure)
            rougeL_scores.append(scores['rougeL'].fmeasure)

    

avg_test_loss = test_loss / len(test_loader)
accuracy = correct / total

print(f"\nРезультаты на тестовом датасете:")
print(f"Средние потери: {avg_test_loss:.4f}")
print(f"Точность (accuracy): {accuracy:.4f}")
print(f"ROUGE-1 F1: {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2 F1: {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L F1: {np.mean(rougeL_scores):.4f}")

Контекст                  | Истинное        | Предсказано     | Сгенерировано  
------------------------------------------------------------------------------------------
please tell me thats somewhere close to | did             | i              
please tell me thats somewhere close to california!!! | not             | i did          
it                        | really          | i did not      
it was                    | see             | i did not really
it was a                  | that            | i did not really see
it was a sleepless        | coming          | i did not really see that
logging                   | totally         | i              
logging out.              | have            | i totally      
logging out. i            | like...         | i totally have 
logging out. i need       | four            | i totally have like...
Контекст                  | Истинное        | Предсказано     | Сгенерировано  
----------------------------------------------------------------

In [107]:
import importlib
import src.data_utils
import src.next_token_utils
importlib.reload(src.next_token_utils)

<module 'src.next_token_utils' from '/Users/irinamasloed/ml_hw/src/next_token_utils.py'>